In [43]:
#This notebook generates 10 repeated 80/20 train-test splits from the master dataset
import os
import numpy as np
import pandas as pd

TARGET = "EnergyPoor"

drop_cols_mepi_other = [
    "hv000CountryCode",
    "hhidCaseIdentification",
    "hv001ClusterNumber",
    "hv002HouseholdNumber",
    "hv005SimplilingWeight",
    "weight",
    "hv206HasElectricity",
    "hv226CookingFuel",
    "hv241FoodCookedInHouse",
    "hv242SeparateKitchenYesNo",
    "hv209HasRefrigerator",
    "hv221HasLandLine",
    "hv243aHasMobilePhone",
    "hv208HasTelevision",
    "Depr_Elec",
    "Depr_CleanFuel",
    "Depr_Refrigerator",
    "Depr_TV",
    "Depr_Phone",
    "Depr_Kitchen",
    "MEPI_score",
]

#normalize to numeric first
cols_to_numeric = [
    "hv024RegionDivision",
    "hv025TypeOfResidence",
    "hv219SexOfHead",
    "hv270WealthIndex",  
    "hv106_01EducationOfHead",
    "hv115_01MaritalStatus",
    "v717Occupation",
    "745aHouseOwnership",
]

In [45]:
def restore_ml_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # numeric types
    df["hv220AgeOfHead"] = df["hv220AgeOfHead"].astype("int8")
    df["hv009FamilySize"] = df["hv009FamilySize"].astype("int8")
    df["hv216HouseSize"] = df["hv216HouseSize"].astype("float64")
    df["hv014NoOfChildren"] = df["hv014NoOfChildren"].astype("int8")

    # categorical
    cat_cols = [
        "hv024RegionDivision",
        "hv025TypeOfResidence",
        "hv219SexOfHead",
        "hv270WealthIndex",
        "hv106_01EducationOfHead",
        "hv115_01MaritalStatus",
        "v717Occupation",
        "745aHouseOwnership",
        "hv247HasBankAccount",
        "v714CurrentlyWorking",
    ]
    existing_cat_cols = [c for c in cat_cols if c in df.columns]
    df[existing_cat_cols] = df[existing_cat_cols].astype("category")

    # binary/int labels
    int8_cols = ["EnergyPoor"]
    existing_int8_cols = [c for c in int8_cols if c in df.columns]
    df[existing_int8_cols] = df[existing_int8_cols].astype("int8")

    return df


In [46]:
def build_country_training_df(
    parquet_path: str,
    target: str,
    cols_to_numeric: list,
    drop_cols: list,
) -> pd.DataFrame:
    # 1) Load
    df_master = pd.read_parquet(parquet_path, engine="pyarrow")

    # 2) Convert selected columns to numeric safely (handles "Missing" strings)
    for col in cols_to_numeric:
        if col in df_master.columns:
            df_master[col] = pd.to_numeric(df_master[col], errors="coerce")

    # 3) Restore ML dtypes (category / int8 etc.)
    df_restore = restore_ml_dtypes(df_master)

    # 4) Make sure target exists
    if target not in df_restore.columns:
        raise ValueError(f"Target '{target}' not found in {parquet_path}")

    # 5) Build X cols by dropping unwanted + target
    drop_set = set(drop_cols + [target])
    X_cols = [c for c in df_restore.columns if c not in drop_set]

    # 6) Final df (features + target)
    df_ready = df_restore[X_cols + [target]].copy()

    return df_ready


In [47]:
country_files = {
    "MYANMAR": "MYANMAR/DATA/4.MM_MASTER_DATASET.parquet",
}

dfs_ready = {}

for country, path in country_files.items():
    df_ready = build_country_training_df(
        parquet_path=path,
        target=TARGET,
        cols_to_numeric=cols_to_numeric,
        drop_cols=drop_cols_mepi_other,
    )
    dfs_ready[country] = df_ready
    print(country, df_ready.shape, "target counts:", df_ready[TARGET].value_counts(dropna=False).to_dict())

# to access:
df_mm = dfs_ready["MYANMAR"]

MYANMAR (12417, 16) target counts: {1: 9664, 0: 2753}


In [48]:
print("\nDtypes:")
print(df_mm.dtypes)


Dtypes:
hv270WealthIndex             category
hv024RegionDivision          category
hv025TypeOfResidence         category
hv219SexOfHead               category
hv220AgeOfHead                   int8
hv106_01EducationOfHead      category
hv115_01MaritalStatus        category
hv009FamilySize                  int8
hv247HasBankAccount          category
hv216HouseSize                float64
hv014NoOfChildren                int8
v714CurrentlyWorking         category
v717Occupation               category
745aHouseOwnership           category
HouseholdClusterElevation     float64
EnergyPoor                       int8
dtype: object


In [49]:
import os

base_path = r"MYANMAR/DATA/PAIRTEST10"

countries = ["MYANMAR"]

for c in countries:
    os.makedirs(f"{base_path}/{c}", exist_ok=True)


In [50]:
from sklearn.model_selection import StratifiedShuffleSplit

def create_and_save_splits(df, country_name, label_col, base_path, n_splits=10, test_size=0.2):
    
    sss = StratifiedShuffleSplit(
        n_splits=n_splits,
        test_size=test_size,
        random_state=42
    )
    
    for i, (train_idx, test_idx) in enumerate(sss.split(df, df[label_col])):
        
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()
        
        split_num = i + 1
        
        train_path = f"{base_path}/split{split_num}_train.parquet"
        test_path = f"{base_path}/split{split_num}_test.parquet"
        
        train_df.to_parquet(train_path)
        test_df.to_parquet(test_path)
        
        print(f"{country_name} Split {split_num} saved.")


In [51]:
LABEL = "EnergyPoor"
base_path = r"MYANMAR/DATA/PAIRTEST10"

create_and_save_splits(df_mm,"MYANMAR", LABEL, base_path)

MYANMAR Split 1 saved.
MYANMAR Split 2 saved.
MYANMAR Split 3 saved.
MYANMAR Split 4 saved.
MYANMAR Split 5 saved.
MYANMAR Split 6 saved.
MYANMAR Split 7 saved.
MYANMAR Split 8 saved.
MYANMAR Split 9 saved.
MYANMAR Split 10 saved.
